# 🔥 ApiKala — Harmonisasi MODIS–VIIRS
**NASA Space Apps Challenge 2026 · Tim Occupy Mars (Bandung)**

Notebook ini mengunduh titik api NASA FIRMS untuk Indonesia (MODIS 2000–sekarang, VIIRS S-NPP / NOAA-20 / NOAA-21),
menyamakan satuannya ("setara MODIS"), menguji hasilnya, lalu mengekspor data kecil untuk web app.

**Cara pakai:** `Runtime → Run all`. Saat diminta, tempel MAP_KEY FIRMS Anda. Di akhir, file `data_web.zip` terunduh otomatis.

## 1. MAP_KEY FIRMS
Kunci tidak disimpan di notebook (pakai `getpass`).

In [ ]:
from getpass import getpass
MAP_KEY = getpass("Tempel MAP_KEY FIRMS lalu Enter: ").strip()
print("MAP_KEY terisi:", len(MAP_KEY), "karakter")

## 2. Kode pipeline
Kode yang sama ada di repo GitHub: `pipeline/harmonisasi.py`.

In [ ]:
%%writefile harmonisasi.py
"""
ApiKala — pipeline harmonisasi titik api MODIS–VIIRS (Indonesia)
NASA Space Apps Challenge 2026 · Tim Occupy Mars

Alur:
  1. UNDUH   : NASA FIRMS Country Yearly Summary (MODIS C6.1, VIIRS S-NPP, NOAA-20, NOAA-21);
               tahun yang belum ada arsipnya (mis. tahun berjalan) diambil lewat FIRMS API (MAP_KEY).
  2. SARING  : MODIS confidence >= 30; VIIRS confidence nominal/high; type = 0 (vegetasi) bila ada.
  3. AGREGAT : hitung titik per sel grid 0,25° × bulan × sensor.
  4. HARMONISASI (target = satuan "setara MODIS Terra+Aqua"):
       - Nov 2000–Jun 2002 hanya Terra  → dikali k_TA = Σ(Terra+Aqua)/ΣTerra (2003–2011)
       - bulan tanpa S-NPP              → VIIRS setara S-NPP = NOAA-20 × k_N20 (atau NOAA-21 × k_N21)
       - 2012 ke atas                   → H = VIIRS × k[pulau, bulan-kalender]
         k = ΣMODIS / ΣVIIRS pada tahun tumpang tindih (default 2012–2021, sebelum orbit
         Terra/Aqua bergeser), dengan selang kepercayaan bootstrap per tahun.
  5. VALIDASI: leave-one-year-out — k dihitung tanpa tahun y lalu dipakai menebak MODIS tahun y.
  6. EKSPOR  : web/data/grid.json + meta.json (dibaca web app).
"""
from __future__ import annotations

import io
import json
import os
import time
from datetime import date, timedelta

import numpy as np
import pandas as pd

FIRMS = "https://firms.modaps.eosdis.nasa.gov"
SUMBER = {
    # kode : (folder arsip tahunan, tahun mulai, sumber API SP, sumber API NRT)
    "MODIS": ("modis", 2000, "MODIS_SP", "MODIS_NRT"),
    "SNPP": ("viirs-snpp", 2012, "VIIRS_SNPP_SP", "VIIRS_SNPP_NRT"),
    "N20": ("viirs-jpss1", 2018, "VIIRS_NOAA20_SP", "VIIRS_NOAA20_NRT"),
    "N21": ("viirs-jpss2", 2024, None, "VIIRS_NOAA21_NRT"),
}
BBOX_IDN = "94,-11.5,141.5,6.5"
BULAN0 = (2000, 11)  # indeks bulan 0 = Nov 2000 (awal MODIS Terra)
AQUA_PENUH = (2002, 7)  # Aqua mulai Juli 2002


def idx_bulan(tahun, bulan):
    return (np.asarray(tahun) - BULAN0[0]) * 12 + (np.asarray(bulan) - BULAN0[1])


def label_bulan(i):
    t, b = divmod(BULAN0[1] - 1 + int(i), 12)
    return f"{BULAN0[0] + t}-{b + 1:02d}"


# ------------------------------------------------------------------ 1. UNDUH
class Pengunduh:
    def __init__(self, map_key: str | None, cache="cache", log=print):
        import requests  # tersedia di Colab

        self.s = requests.Session()
        self.key = map_key
        self.cache = cache
        self.log = log
        self.catatan = []  # asal-usul tiap potong data (dicatat di meta.json)
        os.makedirs(cache, exist_ok=True)

    def _get(self, url, timeout=180, coba=3):
        for i in range(coba):
            try:
                r = self.s.get(url, timeout=timeout)
                if r.status_code == 200:
                    return r.text
                if r.status_code in (400, 401, 403, 404):
                    return None
            except Exception as e:  # jaringan putus, dsb.
                self.log(f"   ! {e.__class__.__name__}, ulangi ({i + 1}/{coba})")
            time.sleep(3 * (i + 1))
        return None

    @staticmethod
    def _valid(txt):
        return txt is not None and txt.lstrip().lower().startswith("latitude")

    def tahunan(self, kode, tahun):
        folder = SUMBER[kode][0]
        url = f"{FIRMS}/data/country/{folder}/{tahun}/{folder}_{tahun}_Indonesia.csv"
        txt = self._get(url)
        return txt if self._valid(txt) else None

    def api(self, sumber, mulai: date, hari: int):
        if not (self.key and sumber):
            return None
        tgl = mulai.isoformat()
        for url in (
            f"{FIRMS}/api/country/csv/{self.key}/{sumber}/IDN/{hari}/{tgl}",
            f"{FIRMS}/api/area/csv/{self.key}/{sumber}/{BBOX_IDN}/{hari}/{tgl}",
        ):
            txt = self._get(url, timeout=120)
            if self._valid(txt):
                return txt
        return None

    def api_tahun(self, kode, tahun, sampai: date):
        """Satu tahun lewat API per jendela 5 hari: coba SP dulu, lalu NRT."""
        _, _, sp, nrt = SUMBER[kode]
        d, akhir = date(tahun, 1, 1), min(date(tahun, 12, 31), sampai)
        potong, n_sp, n_nrt = [], 0, 0
        while d <= akhir:
            hari = min(5, (akhir - d).days + 1)
            txt = self.api(sp, d, hari)
            if txt is not None and txt.count("\n") > 1:
                n_sp += 1
            else:
                txt = self.api(nrt, d, hari)
                n_nrt += txt is not None
            if txt is not None:
                potong.append(pd.read_csv(io.StringIO(txt)))
            d += timedelta(days=hari)
            time.sleep(0.3)
        self.catatan.append(dict(sensor=kode, tahun=tahun, asal="API", jendela_SP=n_sp, jendela_NRT=n_nrt))
        return pd.concat(potong, ignore_index=True) if potong else None


# ------------------------------------------------------------------ 2–3. SARING & AGREGAT
class Grid:
    def __init__(self, cells: dict):
        self.res, self.lon0, self.lat0 = cells["res"], cells["lon0"], cells["lat0"]
        self.ncol, self.nrow = cells["ncol"], cells["nrow"]
        self.sel_prov = {int(k): v for k, v in cells["sel"].items()}
        self.prov_pulau = cells["pulau_provinsi"]
        self.pulau = cells["pulau"]

    def sel(self, lon, lat):
        c = np.floor((np.asarray(lon) - self.lon0) / self.res).astype(int)
        r = np.floor((np.asarray(lat) - self.lat0) / self.res).astype(int)
        ok = (c >= 0) & (c < self.ncol) & (r >= 0) & (r < self.nrow)
        return np.where(ok, r * self.ncol + c, -1)

    def pulau_sel(self, sel):
        return np.array([self.prov_pulau[self.sel_prov[s]] for s in sel])


def saring_agregat(df: pd.DataFrame, kode: str, grid: Grid) -> pd.DataFrame:
    """Kembalikan tabel (sel, m, sensor, n). Sensor: MT/MA untuk MODIS Terra/Aqua."""
    if df is None or df.empty:
        return pd.DataFrame(columns=["sel", "m", "sensor", "n"])
    df = df.rename(columns=str.lower)
    awal = len(df)
    conf = df["confidence"]
    if kode == "MODIS":
        ok = pd.to_numeric(conf, errors="coerce").fillna(0) >= 30
    else:
        ok = conf.astype(str).str.strip().str.lower().str[0].isin(["n", "h"])
    if "type" in df:
        ok &= pd.to_numeric(df["type"], errors="coerce").fillna(0).astype(int) == 0
    df = df[ok]
    sel = grid.sel(df["longitude"].to_numpy(), df["latitude"].to_numpy())
    di_idn = np.isin(sel, list(grid.sel_prov))
    df = df[di_idn].assign(sel=sel[di_idn])
    tgl = pd.to_datetime(df["acq_date"])
    df = df.assign(m=idx_bulan(tgl.dt.year.to_numpy(), tgl.dt.month.to_numpy()))
    if kode == "MODIS":
        sat = df["satellite"].astype(str).str.upper().str[0]
        df = df.assign(sensor=np.where(sat == "T", "MT", "MA"))
    else:
        df = df.assign(sensor=kode)
    out = df.groupby(["sel", "m", "sensor"]).size().rename("n").reset_index()
    out.attrs["saring"] = (awal, int(ok.sum()), len(df))
    return out


def kumpulkan(map_key, cells, tahun_akhir=None, cache="cache", log=print, offline_dir=None):
    """Unduh + saring + agregat semua sensor. Hasil per (sensor,tahun) disimpan di cache."""
    grid = Grid(cells)
    hari_ini = date.today()
    os.makedirs(cache, exist_ok=True)
    tahun_akhir = tahun_akhir or hari_ini.year
    dl = None if offline_dir else Pengunduh(map_key, cache, log)
    potong, catatan = [], []
    for kode, (folder, t0, _, _) in SUMBER.items():
        for th in range(t0, tahun_akhir + 1):
            fc = os.path.join(cache, f"agg_{kode}_{th}.csv")
            if os.path.exists(fc) and th < hari_ini.year:
                agg = pd.read_csv(fc)
                potong.append(agg)
                catatan.append(dict(sensor=kode, tahun=th, asal="cache", baris=int(agg["n"].sum())))
                continue
            df, asal = None, None
            if offline_dir:
                p = os.path.join(offline_dir, f"{folder}_{th}_Indonesia.csv")
                if os.path.exists(p):
                    df, asal = pd.read_csv(p), "arsip-tahunan"
            else:
                txt = dl.tahunan(kode, th)
                if txt is not None:
                    df, asal = pd.read_csv(io.StringIO(txt)), "arsip-tahunan"
                elif th >= hari_ini.year - 1:
                    log(f"   {kode} {th}: arsip tahunan belum ada → FIRMS API (per 5 hari)")
                    df, asal = dl.api_tahun(kode, th, hari_ini), "API"
                else:
                    log(f"   {kode} {th}: arsip tahunan TIDAK ditemukan (dilewati)")
            if df is None:
                log(f"   {kode} {th}: tidak ada data")
                catatan.append(dict(sensor=kode, tahun=th, asal="tidak-ada", baris=0))
                continue
            agg = saring_agregat(df, kode, grid)
            awal, lolos, idn = agg.attrs.get("saring", (len(df), 0, 0))
            log(f"   {kode} {th}: {awal:>9,} titik → lolos saring {lolos:>9,} → di sel Indonesia {idn:>9,}  ({asal})")
            agg.to_csv(fc, index=False)
            potong.append(agg)
            catatan.append(dict(sensor=kode, tahun=th, asal=asal, baris_mentah=int(awal),
                                lolos_saring=int(lolos), baris=int(idn)))
    if dl:
        catatan += dl.catatan
    tabel = pd.concat(potong, ignore_index=True)
    return tabel, catatan, grid


# ------------------------------------------------------------------ 4. HARMONISASI
def rasio(a, b):
    a, b = float(np.sum(a)), float(np.sum(b))
    return a / b if b > 0 else np.nan


def bootstrap_k(per_tahun: pd.DataFrame, num="M", den="V", n=500, rng=None):
    """CI 95% rasio ΣM/ΣV dengan resampling tahun (menangkap variasi antar-tahun)."""
    rng = rng or np.random.default_rng(42)
    a, b = per_tahun[num].to_numpy(float), per_tahun[den].to_numpy(float)
    if len(a) < 3 or b.sum() == 0:
        return np.nan, np.nan
    idx = rng.integers(0, len(a), size=(n, len(a)))
    ks = a[idx].sum(1) / np.maximum(b[idx].sum(1), 1e-9)
    return float(np.percentile(ks, 2.5)), float(np.percentile(ks, 97.5))


def harmonisasi(tabel: pd.DataFrame, grid: Grid, overlap=(2012, 2021), min_viirs=300, log=print):
    # tabel lebar: sel × m × sensor
    lebar = tabel.pivot_table(index=["sel", "m"], columns="sensor", values="n", aggfunc="sum", fill_value=0)
    for c in ["MT", "MA", "SNPP", "N20", "N21"]:
        if c not in lebar:
            lebar[c] = 0
    lebar = lebar.reset_index()
    lebar["pulau"] = grid.pulau_sel(lebar["sel"].to_numpy())
    lebar["tahun"] = BULAN0[0] + (lebar["m"] + BULAN0[1] - 1) // 12
    lebar["bulan"] = (lebar["m"] + BULAN0[1] - 1) % 12 + 1

    # total nasional per bulan → tahu bulan mana tiap sensor aktif
    nas = lebar.groupby("m")[["MT", "MA", "SNPP", "N20", "N21"]].sum()

    # (a) Terra-only → Terra+Aqua
    m_aqua = int(idx_bulan(*AQUA_PENUH))
    ref = lebar[(lebar["tahun"] >= 2003) & (lebar["tahun"] <= 2011)]
    k_TA = rasio(ref["MT"] + ref["MA"], ref["MT"])
    lebar["M"] = (lebar["MT"] + lebar["MA"]).astype(float)
    pre = lebar["m"] < m_aqua
    lebar.loc[pre, "M"] = lebar.loc[pre, "MT"] * k_TA

    # (b) NOAA-20 / NOAA-21 → setara S-NPP (bulan yang ketiganya aktif)
    def k_jembatan(kol):
        bulan_ok = nas.index[(nas["SNPP"] > 0) & (nas[kol] > 0)]
        sub = lebar[lebar["m"].isin(bulan_ok)]
        return rasio(sub["SNPP"], sub[kol]), int(len(bulan_ok))

    k_N20, nb20 = k_jembatan("N20")
    k_N21, nb21 = k_jembatan("N21")
    # bulan S-NPP dipakai hanya bila cakupannya penuh (S-NPP berhenti 2 Nov 2026: bulan itu → NOAA-20)
    snpp_aktif = set(nas.index[(nas["SNPP"] > 0) & ((nas["N20"] == 0) | (nas["SNPP"] >= 0.5 * nas["N20"]))])
    n20_aktif = set(nas.index[nas["N20"] > 0])
    sumber_v = {}
    V = lebar["SNPP"].astype(float).to_numpy().copy()
    for m in sorted(set(lebar["m"])):
        if m in snpp_aktif:
            sumber_v[m] = "SNPP"
        elif m in n20_aktif and np.isfinite(k_N20):
            sumber_v[m] = "N20"
        elif nas.loc[m, "N21"] > 0 and np.isfinite(k_N21):
            sumber_v[m] = "N21"
    mm = lebar["m"].to_numpy()
    for kol, k in (("N20", k_N20), ("N21", k_N21)):
        pakai = np.array([sumber_v.get(m) == kol for m in mm])
        V[pakai] = lebar[kol].to_numpy()[pakai] * k
    lebar["V"] = V

    # (c) faktor utama k[pulau, bulan] = ΣM / ΣV pada tahun tumpang tindih
    ov = lebar[(lebar["tahun"] >= overlap[0]) & (lebar["tahun"] <= overlap[1])]
    rng = np.random.default_rng(42)
    baris_k = []
    k_nas_bulan = {}
    for b in range(1, 13):
        sub = ov[ov["bulan"] == b]
        k_nas_bulan[b] = rasio(sub["M"], sub["V"])
    k_nas = rasio(ov["M"], ov["V"])
    for p_i, p in enumerate(grid.pulau):
        sp = ov[ov["pulau"] == p_i]
        k_pulau = rasio(sp["M"], sp["V"]) if sp["V"].sum() >= min_viirs else k_nas
        for b in range(1, 13):
            sb = sp[sp["bulan"] == b]
            pt = sb.groupby("tahun")[["M", "V"]].sum()
            if sb["V"].sum() >= min_viirs:
                k, metode = rasio(sb["M"], sb["V"]), "pulau-bulan"
            elif sp["V"].sum() >= min_viirs:
                k, metode = k_pulau, "pulau-tahunan"
                pt = sp.groupby("tahun")[["M", "V"]].sum()
            else:
                k, metode = k_nas_bulan[b], "nasional-bulan"
                pt = ov[ov["bulan"] == b].groupby("tahun")[["M", "V"]].sum()
            lo, hi = bootstrap_k(pt, rng=rng)
            baris_k.append(dict(pulau=p, bulan=b, k=round(k, 4), ci_lo=round(lo, 4), ci_hi=round(hi, 4),
                                metode=metode, n_viirs=int(sb["V"].sum()), n_modis=int(round(sb["M"].sum()))))
    ktab = pd.DataFrame(baris_k)
    K = {(r.pulau, r.bulan): r.k for r in ktab.itertuples()}
    m_v0 = min(snpp_aktif) if snpp_aktif else 10**9
    kk = np.array([K[(grid.pulau[p], b)] for p, b in zip(lebar["pulau"], lebar["bulan"])])
    lebar["H"] = np.where(lebar["m"] >= m_v0, lebar["V"] * kk, lebar["M"])

    log(f"   k_TA (Terra→Terra+Aqua) = {k_TA:.3f} | k_N20→SNPP = {k_N20:.3f} ({nb20} bln) | "
        f"k_N21→SNPP = {k_N21:.3f} ({nb21} bln) | k nasional = {k_nas:.3f}")
    info = dict(k_TA=k_TA, k_N20=k_N20, k_N21=k_N21, k_nasional=k_nas, overlap=list(overlap),
                bulan_mulai_viirs=label_bulan(m_v0), bulan_aqua=label_bulan(m_aqua),
                sumber_viirs_per_bulan={label_bulan(m): s for m, s in sumber_v.items() if s != "SNPP"})
    return lebar, ktab, info


# ------------------------------------------------------------------ 5. VALIDASI
def validasi(lebar: pd.DataFrame, grid: Grid, overlap=(2012, 2021)):
    """Leave-one-year-out pada tingkat pulau × bulan, dibandingkan dengan VIIRS mentah."""
    ov = lebar[(lebar["tahun"] >= overlap[0]) & (lebar["tahun"] <= overlap[1]) & (lebar["m"] >= lebar.loc[lebar["SNPP"] > 0, "m"].min())]
    agg = ov.groupby(["pulau", "tahun", "bulan"])[["M", "V"]].sum().reset_index()
    pred = []
    for y in sorted(agg["tahun"].unique()):
        latih, uji = agg[agg["tahun"] != y], agg[agg["tahun"] == y]
        kb = latih.groupby(["pulau", "bulan"])[["M", "V"]].sum()
        kp = latih.groupby("pulau")[["M", "V"]].sum()
        for r in uji.itertuples():
            mv = kb.loc[(r.pulau, r.bulan)] if (r.pulau, r.bulan) in kb.index else None
            if mv is not None and mv["V"] >= 300:
                k = mv["M"] / mv["V"]
            else:
                k = kp.loc[r.pulau, "M"] / max(kp.loc[r.pulau, "V"], 1)
            pred.append(dict(pulau=grid.pulau[r.pulau], tahun=r.tahun, bulan=r.bulan,
                             modis=r.M, viirs_mentah=r.V, harmonis=r.V * k))
    p = pd.DataFrame(pred)

    def metrik(d):
        d = d[d["modis"] >= 20]
        if len(d) < 3:
            return None
        out = {}
        for nama, col in (("harmonis", "harmonis"), ("viirs_mentah", "viirs_mentah")):
            e = d[col] - d["modis"]
            ss = ((d["modis"] - d["modis"].mean()) ** 2).sum()
            out[nama] = dict(
                r=round(float(np.corrcoef(d["modis"], d[col])[0, 1]), 3),
                r2=round(float(1 - (e ** 2).sum() / ss), 3) if ss > 0 else None,
                bias_pct=round(float(100 * e.sum() / d["modis"].sum()), 1),
                mape_pct=round(float(100 * (e.abs() / d["modis"]).median()), 1),
            )
        out["n_bulan"] = int(len(d))
        return out

    hasil = {"Indonesia (pulau×bulan)": metrik(p)}
    for nama in grid.pulau:
        hasil[nama] = metrik(p[p["pulau"] == nama])
    nas = p.groupby(["tahun", "bulan"])[["modis", "viirs_mentah", "harmonis"]].sum().reset_index()
    hasil["Indonesia (nasional×bulan)"] = metrik(nas.assign(pulau="Indonesia"))
    return hasil, p


# ------------------------------------------------------------------ 6. EKSPOR
def _bersih(x):
    """NaN/inf → null, tipe numpy → tipe Python (agar JSON valid untuk peramban)."""
    if isinstance(x, dict):
        return {str(k): _bersih(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_bersih(v) for v in x]
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (float, np.floating)):
        return float(x) if np.isfinite(x) else None
    return x


def ekspor(lebar, ktab, info, val, catatan, cells, keluar="web/data", filters=None):
    os.makedirs(keluar, exist_ok=True)
    n_bulan = int(lebar["m"].max()) + 1
    # grid.json: per sel, deret jarang [m, M, V, m, M, V, ...] (H dihitung ulang di peramban)
    g = {}
    for sel, d in lebar[(lebar["M"] > 0) | (lebar["V"] > 0)].sort_values("m").groupby("sel"):
        arr = np.column_stack([d["m"], np.round(d["M"], 1), np.round(d["V"], 1)]).ravel().tolist()
        g[str(int(sel))] = [int(x) if float(x).is_integer() else x for x in arr]
    json.dump(dict(bulan0=label_bulan(0), n_bulan=n_bulan, sel=g),
              open(os.path.join(keluar, "grid.json"), "w"), separators=(",", ":"))
    ksel = {p: [None] * 12 for p in ktab["pulau"].unique()}
    for r in ktab.itertuples():
        ksel[r.pulau][r.bulan - 1] = r.k
    meta = dict(
        dibuat=pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M UTC"),
        tanggal_data=date.today().isoformat(),
        bulan_awal=label_bulan(0), bulan_akhir=label_bulan(n_bulan - 1),
        k=ksel, k_detail=ktab.to_dict("records"),
        k_TA=round(info["k_TA"], 4), k_N20=round(info["k_N20"], 4) if np.isfinite(info["k_N20"]) else None,
        k_N21=round(info["k_N21"], 4) if np.isfinite(info["k_N21"]) else None,
        k_nasional=round(info["k_nasional"], 4), overlap=info["overlap"],
        bulan_mulai_viirs=info["bulan_mulai_viirs"], bulan_aqua=info["bulan_aqua"],
        sumber_viirs_pengganti=info["sumber_viirs_per_bulan"],
        validasi=val, saringan=filters or {}, asal_data=catatan,
        sumber=["NASA FIRMS — MODIS Collection 6.1 (MCD14) active fire, Terra & Aqua",
                "NASA FIRMS — VIIRS 375 m (VNP14IMG/VJ114IMG/VJ214IMG) S-NPP, NOAA-20, NOAA-21",
                "geoBoundaries IDN ADM1 (CC BY 4.0)"],
    )
    json.dump(_bersih(meta), open(os.path.join(keluar, "meta.json"), "w"), ensure_ascii=False, indent=1,
              allow_nan=False)
    json.dump(cells, open(os.path.join(keluar, "cells.json"), "w"), ensure_ascii=False, separators=(",", ":"))
    ukuran = {f: os.path.getsize(os.path.join(keluar, f)) // 1024 for f in ("grid.json", "meta.json", "cells.json")}
    return ukuran


def jalankan(map_key, cells, keluar="web/data", cache="cache", overlap=(2012, 2021), offline_dir=None, log=print):
    log("1–3. Unduh, saring, agregat …")
    tabel, catatan, grid = kumpulkan(map_key, cells, cache=cache, log=log, offline_dir=offline_dir)
    log("4. Harmonisasi …")
    lebar, ktab, info = harmonisasi(tabel, grid, overlap=overlap, log=log)
    log("5. Validasi leave-one-year-out …")
    val, pred = validasi(lebar, grid, overlap=overlap)
    log(json.dumps(val.get("Indonesia (pulau×bulan)"), indent=1))
    log("6. Ekspor …")
    filt = {"MODIS": "confidence >= 30, type = 0", "VIIRS": "confidence nominal/high, type = 0",
            "grid": "0,25° (±28 km)", "wilayah": "sel yang dipetakan ke 34 provinsi (geoBoundaries)"}
    ukuran = ekspor(lebar, ktab, info, val, catatan, cells, keluar=keluar, filters=filt)
    log(f"   selesai: {ukuran} KB")
    return lebar, ktab, info, val, pred

In [ ]:
%%writefile cells.json
{"res":0.25,"lon0":94.0,"lat0":-11.5,"ncol":190,"nrow":72,"provinsi":["Aceh","Bali","Kep. Bangka Belitung","Banten","Bengkulu","Jawa Tengah","Kalimantan Tengah","Sulawesi Tengah","Jawa Timur","Kalimantan Timur","Nusa Tenggara Timur","Gorontalo","DKI Jakarta","Jambi","Lampung","Maluku","Kalimantan Utara","Maluku Utara","Sulawesi Utara","Sumatera Utara","Papua","Riau","Kepulauan Riau","Kalimantan Selatan","Sulawesi Selatan","Sumatera Selatan","Sulawesi Tenggara","DI Yogyakarta","Jawa Barat","Kalimantan Barat","Nusa Tenggara Barat","Papua Barat","Sulawesi Barat","Sumatera Barat"],"provinsi_en":["Aceh","Bali","Bangka-Belitung Islands","Banten","Bengkulu","Central Java","Central Kalimantan","Central Sulawesi","East Java","East Kalimantan","East Nusa Tenggara","Gorontalo","Jakarta Special Capital Region","Jambi","Lampung","Maluku","North Kalimantan","North Maluku","North Sulawesi","North Sumatra","Papua","Riau","Riau Islands","South Kalimantan","South Sulawesi","South Sumatra","Southeast Sulawesi","Special Region of Yogyakarta","West Java","West Kalimantan","West Nusa Tenggara","West Papua","West Sulawesi","West Sumatra"],"pulau":["Sumatra","Jawa-Bali","Nusa Tenggara","Kalimantan","Sulawesi","Maluku","Papua"],"pulau_provinsi":[0,1,0,1,0,1,3,4,1,3,2,4,1,0,0,5,3,5,4,0,6,0,0,3,4,0,4,1,1,3,2,6,4,0],"sel":{"304":10,"305":10,"306":10,"490":10,"491":10,"493":10,"494":10,"495":10,"496":10,"497":10,"498":10,"674":10,"680":10,"681":10,"682":10,"684":10,"685":10,"686":10,"687":10,"688":10,"689":10,"863":10,"864":10,"865":10,"866":10,"867":10,"870":10,"871":10,"872":10,"876":10,"877":10,"878":10,"879":10,"880":10,"881":10,"1052":10,"1053":10,"1054":10,"1055":10,"1056":10,"1057":10,"1058":10,"1061":10,"1066":10,"1067":10,"1068":10,"1069":10,"1070":10,"1071":10,"1072":10,"1073":10,"1239":10,"1240":10,"1241":10,"1242":10,"1243":10,"1244":10,"1245":10,"1246":10,"1247":10,"1257":10,"1258":10,"1259":10,"1260":10,"1261":10,"1262":10,"1263":10,"1264":10,"1429":10,"1430":10,"1431":10,"1432":10,"1433":10,"1434":10,"1435":10,"1436":10,"1437":10,"1447":10,"1448":10,"1449":10,"1450":10,"1451":10,"1452":10,"1453":10,"1454":10,"1611":30,"1612":30,"1619":10,"1620":10,"1621":10,"1622":10,"1623":10,"1624":10,"1625":10,"1626":10,"1638":10,"1639":10,"1640":10,"1641":10,"1642":10,"1643":10,"1644":10,"1645":10,"1707":20,"1708":20,"1794":1,"1797":30,"1798":30,"1799":30,"1800":30,"1801":30,"1802":30,"1803":30,"1804":30,"1805":30,"1806":30,"1807":30,"1808":30,"1809":30,"1810":10,"1811":10,"1812":10,"1813":10,"1814":10,"1815":10,"1816":10,"1817":10,"1818":10,"1819":10,"1820":10,"1821":10,"1829":10,"1830":10,"1831":10,"1832":10,"1833":10,"1834":10,"1835":10,"1896":20,"1897":20,"1898":20,"1979":8,"1980":8,"1981":8,"1982":8,"1983":1,"1984":1,"1985":1,"1986":1,"1987":30,"1988":30,"1989":30,"1990":30,"1991":30,"1992":30,"1993":30,"1994":30,"1995":30,"1996":30,"1997":30,"1998":30,"1999":30,"2000":30,"2001":10,"2002":10,"2003":10,"2004":10,"2005":10,"2006":10,"2007":10,"2008":10,"2009":10,"2010":10,"2011":10,"2012":10,"2013":10,"2014":10,"2015":10,"2017":10,"2022":10,"2023":10,"2024":10,"2025":10,"2085":20,"2086":20,"2087":20,"2088":20,"2160":8,"2161":8,"2162":8,"2163":8,"2164":8,"2165":8,"2166":8,"2167":8,"2168":8,"2169":8,"2170":8,"2171":8,"2172":8,"2173":1,"2174":1,"2175":1,"2176":1,"2177":30,"2178":30,"2179":30,"2180":30,"2181":30,"2182":30,"2183":30,"2184":30,"2185":30,"2186":30,"2187":30,"2188":30,"2189":30,"2190":30,"2191":10,"2192":10,"2193":10,"2194":10,"2195":10,"2196":10,"2197":10,"2198":10,"2199":10,"2200":10,"2201":10,"2202":10,"2203":10,"2204":10,"2205":10,"2206":10,"2207":10,"2208":10,"2209":10,"2210":10,"2211":10,"2212":10,"2213":10,"2214":10,"2237":15,"2264":20,"2265":20,"2266":20,"2267":20,"2268":20,"2269":20,"2274":20,"2275":20,"2276":20,"2277":20,"2345":27,"2346":27,"2347":5,"2348":8,"2349":8,"2350":8,"2351":8,"2352":8,"2353":8,"2354":8,"2355":8,"2356":8,"2357":8,"2358":8,"2359":8,"2360":8,"2361":8,"2362":1,"2363":1,"2364":1,"2365":1,"2366":1,"2367":30,"2368":30,"2369":30,"2370":30,"2371":30,"2372":30,"2373":30,"2374":30,"2375":30,"2376":30,"2377":30,"2378":30,"2379":30,"2380":30,"2381":10,"2382":10,"2383":10,"2384":10,"2385":10,"2386":10,"2387":10,"2388":10,"2389":10,"2390":10,"2391":10,"2392":10,"2393":10,"2394":10,"2395":10,"2396":10,"2397":10,"2398":10,"2399":10,"2400":10,"2401":10,"2402":10,"2403":10,"2404":10,"2405":10,"2412":15,"2413":15,"2414":15,"2415":15,"2416":15,"2417":15,"2418":15,"2419":15,"2420":15,"2422":15,"2423":15,"2426":15,"2427":15,"2428":15,"2453":20,"2454":20,"2455":20,"2456":20,"2457":20,"2458":20,"2459":20,"2460":20,"2461":20,"2462":20,"2463":20,"2464":20,"2465":20,"2466":20,"2467":20,"2533":27,"2534":27,"2535":27,"2536":27,"2537":5,"2538":8,"2539":8,"2540":8,"2541":8,"2542":8,"2543":8,"2544":8,"2545":8,"2546":8,"2547":8,"2548":8,"2549":8,"2550":8,"2551":8,"2552":1,"2553":1,"2554":1,"2555":1,"2556":1,"2557":30,"2558":30,"2559":30,"2560":30,"2561":30,"2562":30,"2563":30,"2564":30,"2565":30,"2566":30,"2567":30,"2568":30,"2569":30,"2570":30,"2571":30,"2573":10,"2574":10,"2575":10,"2576":10,"2577":10,"2578":10,"2580":10,"2581":10,"2582":10,"2583":10,"2584":10,"2585":10,"2586":10,"2587":10,"2588":10,"2589":10,"2590":10,"2591":10,"2592":10,"2593":10,"2594":10,"2595":10,"2596":15,"2597":15,"2598":15,"2599":15,"2600":15,"2601":15,"2602":15,"2603":15,"2604":15,"2605":15,"2606":15,"2607":15,"2608":15,"2609":15,"2610":15,"2611":15,"2612":15,"2613":15,"2614":15,"2616":15,"2617":15,"2618":15,"2619":15,"2644":20,"2645":20,"2646":20,"2647":20,"2648":20,"2649":20,"2650":20,"2651":20,"2652":20,"2653":20,"2654":20,"2655":20,"2656":20,"2657":20,"2714":28,"2715":28,"2716":28,"2717":28,"2718":28,"2719":5,"2720":5,"2721":5,"2722":5,"2723":5,"2724":27,"2725":27,"2726":27,"2727":5,"2728":5,"2729":8,"2730":8,"2731":8,"2732":8,"2733":8,"2734":8,"2735":8,"2736":8,"2737":8,"2738":8,"2739":8,"2740":8,"2741":8,"2742":8,"2744":1,"2745":1,"2753":30,"2754":30,"2755":30,"2756":30,"2760":30,"2772":10,"2774":10,"2775":10,"2777":10,"2778":10,"2779":10,"2781":10,"2782":10,"2783":10,"2784":10,"2786":15,"2787":15,"2788":15,"2789":15,"2790":15,"2791":15,"2792":15,"2793":15,"2794":15,"2795":15,"2796":15,"2798":15,"2801":15,"2802":15,"2803":15,"2804":15,"2805":15,"2807":15,"2808":15,"2809":15,"2810":15,"2834":20,"2835":20,"2836":20,"2837":20,"2838":20,"2839":20,"2840":20,"2841":20,"2842":20,"2843":20,"2844":20,"2845":20,"2846":20,"2847":20,"2899":28,"2900":28,"2901":28,"2902":28,"2903":28,"2904":28,"2905":28,"2906":28,"2907":28,"2908":28,"2909":5,"2910":5,"2911":5,"2912":5,"2913":5,"2914":5,"2915":27,"2916":5,"2917":5,"2918":5,"2919":8,"2920":8,"2921":8,"2922":8,"2923":8,"2924":8,"2925":8,"2926":8,"2927":8,"2928":8,"2929":8,"2930":8,"2931":8,"2932":8,"2942":30,"2943":30,"2944":30,"2958":24,"2960":24,"2961":24,"2967":10,"2968":10,"2976":15,"2977":15,"2978":15,"2979":15,"2980":15,"2981":15,"2982":15,"2983":15,"2984":15,"2985":15,"2988":15,"2991":15,"2992":15,"2993":15,"2994":15,"2996":15,"2997":15,"2998":15,"2999":15,"3000":15,"3001":15,"3025":20,"3026":20,"3027":20,"3028":20,"3029":20,"3030":20,"3031":20,"3032":20,"3033":20,"3034":20,"3035":20,"3036":20,"3037":20,"3088":28,"3089":28,"3090":28,"3091":28,"3092":28,"3093":28,"3094":28,"3095":28,"3096":28,"3097":28,"3098":28,"3099":5,"3100":5,"3101":5,"3102":5,"3103":5,"3104":5,"3105":5,"3106":5,"3107":5,"3108":5,"3109":8,"3110":8,"3111":8,"3112":8,"3113":8,"3114":8,"3115":8,"3116":8,"3117":8,"3118":8,"3119":8,"3120":8,"3121":8,"3122":8,"3123":8,"3126":8,"3127":8,"3128":8,"3132":30,"3133":30,"3134":30,"3136":24,"3146":24,"3147":24,"3148":24,"3149":24,"3150":24,"3151":24,"3152":24,"3167":15,"3169":15,"3170":15,"3171":15,"3172":15,"3173":15,"3174":15,"3177":15,"3178":15,"3179":15,"3182":15,"3183":15,"3186":15,"3187":15,"3188":15,"3189":15,"3190":15,"3191":15,"3192":15,"3202":15,"3215":20,"3216":20,"3217":20,"3218":20,"3219":20,"3220":20,"3221":20,"3222":20,"3223":20,"3224":20,"3225":20,"3226":20,"3227":20,"3275":3,"3276":3,"3277":3,"3278":3,"3279":28,"3280":28,"3281":28,"3282":28,"3283":28,"3284":28,"3285":28,"3286":28,"3287":28,"3288":28,"3289":5,"3290":5,"3291":5,"3292":5,"3293":5,"3294":5,"3295":5,"3296":5,"3297":5,"3298":5,"3299":5,"3300":8,"3301":8,"3302":8,"3303":8,"3304":8,"3305":8,"3306":8,"3307":8,"3308":8,"3309":8,"3310":8,"3311":8,"3312":8,"3313":8,"3314":8,"3315":8,"3316":8,"3317":8,"3318":8,"3319":8,"3325":24,"3326":24,"3327":24,"3330":24,"3335":24,"3336":24,"3337":24,"3338":24,"3340":24,"3341":24,"3367":15,"3368":15,"3369":15,"3370":15,"3371":15,"3378":15,"3379":15,"3380":15,"3381":15,"3382":15,"3390":15,"3391":15,"3392":15,"3393":15,"3407":20,"3408":20,"3409":20,"3410":20,"3411":20,"3412":20,"3413":20,"3414":20,"3415":20,"3416":20,"3417":20,"3418":20,"3464":3,"3465":3,"3466":3,"3467":3,"3468":3,"3469":3,"3470":28,"3471":28,"3472":28,"3473":28,"3474":28,"3475":28,"3476":28,"3477":28,"3478":28,"3479":5,"3480":5,"3481":5,"3482":5,"3483":5,"3484":5,"3485":5,"3486":5,"3487":5,"3488":5,"3489":5,"3490":5,"3491":8,"3492":8,"3493":8,"3494":8,"3495":8,"3496":8,"3497":8,"3498":8,"3499":8,"3500":8,"3501":8,"3502":8,"3503":8,"3504":8,"3505":8,"3506":8,"3507":8,"3508":8,"3509":8,"3516":24,"3517":24,"3519":24,"3520":24,"3521":24,"3525":24,"3526":24,"3527":24,"3557":15,"3558":15,"3559":15,"3560":15,"3561":15,"3562":15,"3569":15,"3570":15,"3571":15,"3572":15,"3579":15,"3580":15,"3581":15,"3582":15,"3583":15,"3597":20,"3598":20,"3599":20,"3600":20,"3601":20,"3602":20,"3603":20,"3604":20,"3605":20,"3606":20,"3607":20,"3608":20,"3653":3,"3654":3,"3655":3,"3656":3,"3657":3,"3658":3,"3659":3,"3660":28,"3661":28,"3662":28,"3663":28,"3664":28,"3665":28,"3666":28,"3667":28,"3668":28,"3669":28,"3670":5,"3671":5,"3672":5,"3673":5,"3674":5,"3675":5,"3676":5,"3677":5,"3678":5,"3679":5,"3680":5,"3681":8,"3682":8,"3683":8,"3684":8,"3685":8,"3686":8,"3687":8,"3688":8,"3689":8,"3690":8,"3694":8,"3695":8,"3696":8,"3708":24,"3709":24,"3710":24,"3711":24,"3714":24,"3715":24,"3716":24,"3717":24,"3751":15,"3752":15,"3759":15,"3760":15,"3761":15,"3769":15,"3770":15,"3771":15,"3772":15,"3773":15,"3787":20,"3788":20,"3789":20,"3790":20,"3791":20,"3792":20,"3793":20,"3794":20,"3795":20,"3796":20,"3797":20,"3798":20,"3844":3,"3845":3,"3846":3,"3847":3,"3848":3,"3849":3,"3850":28,"3851":28,"3852":28,"3853":28,"3854":28,"3855":28,"3856":28,"3857":28,"3858":28,"3866":5,"3867":5,"3868":5,"3869":5,"3870":5,"3898":24,"3899":24,"3905":24,"3906":24,"3914":26,"3915":26,"3922":26,"3943":15,"3944":15,"3950":15,"3959":15,"3960":15,"3961":15,"3962":15,"3963":15,"3964":15,"3976":20,"3977":20,"3978":20,"3979":20,"3980":20,"3981":20,"3982":20,"3983":20,"3984":20,"3985":20,"3986":20,"3987":20,"3988":20,"4031":14,"4032":14,"4033":14,"4034":14,"4035":14,"4036":14,"4037":3,"4038":3,"4039":3,"4040":3,"4041":12,"4042":28,"4043":28,"4044":28,"4045":28,"4046":28,"4047":28,"4048":28,"4055":5,"4056":5,"4057":5,"4064":8,"4094":24,"4095":24,"4096":24,"4103":26,"4104":26,"4105":26,"4109":26,"4110":26,"4111":26,"4112":26,"4113":26,"4133":15,"4134":15,"4143":15,"4144":15,"4145":15,"4146":15,"4149":15,"4150":15,"4151":15,"4152":15,"4153":15,"4154":15,"4166":20,"4167":20,"4168":20,"4169":20,"4170":20,"4171":20,"4172":20,"4173":20,"4174":20,"4175":20,"4176":20,"4177":20,"4178":20,"4220":14,"4221":14,"4222":14,"4223":14,"4224":14,"4225":14,"4226":14,"4227":14,"4228":3,"4229":3,"4230":3,"4231":28,"4232":28,"4233":28,"4234":28,"4244":5,"4245":5,"4246":5,"4247":5,"4253":8,"4254":8,"4255":8,"4261":8,"4281":24,"4282":24,"4283":24,"4284":24,"4285":24,"4286":24,"4293":26,"4294":26,"4295":26,"4298":26,"4299":26,"4300":26,"4302":26,"4331":15,"4332":15,"4333":15,"4334":15,"4335":15,"4336":15,"4339":15,"4340":15,"4341":15,"4342":15,"4343":15,"4344":15,"4355":20,"4356":20,"4357":20,"4358":20,"4359":20,"4360":20,"4361":20,"4362":20,"4363":20,"4364":20,"4365":20,"4366":20,"4367":20,"4402":4,"4403":4,"4404":4,"4409":14,"4410":14,"4411":14,"4412":14,"4413":14,"4414":14,"4415":14,"4416":14,"4417":14,"4418":3,"4419":3,"4434":5,"4435":5,"4436":5,"4443":8,"4444":8,"4445":8,"4450":8,"4451":8,"4452":8,"4465":24,"4466":24,"4467":24,"4468":24,"4470":24,"4471":24,"4472":24,"4473":24,"4474":24,"4475":24,"4476":24,"4481":26,"4482":26,"4483":26,"4484":26,"4485":26,"4486":26,"4487":26,"4488":26,"4489":26,"4490":26,"4491":26,"4514":15,"4515":15,"4521":15,"4522":15,"4523":15,"4524":15,"4525":15,"4526":15,"4527":15,"4530":15,"4531":15,"4532":15,"4533":15,"4544":20,"4545":20,"4546":20,"4547":20,"4548":20,"4549":20,"4550":20,"4551":20,"4552":20,"4553":20,"4554":20,"4555":20,"4556":20,"4557":20,"4591":4,"4592":4,"4593":4,"4594":4,"4599":14,"4600":14,"4601":14,"4602":14,"4603":14,"4604":14,"4605":14,"4606":14,"4607":14,"4608":14,"4641":8,"4642":8,"4652":23,"4654":24,"4655":24,"4656":24,"4657":24,"4658":24,"4660":24,"4661":24,"4662":24,"4663":24,"4664":24,"4665":24,"4666":24,"4670":26,"4671":26,"4672":26,"4673":26,"4674":26,"4675":26,"4676":26,"4677":26,"4678":26,"4679":26,"4680":26,"4681":26,"4682":26,"4704":15,"4705":15,"4711":15,"4712":15,"4713":15,"4714":15,"4715":15,"4716":15,"4717":15,"4720":15,"4721":15,"4722":15,"4723":15,"4733":20,"4734":20,"4735":20,"4736":20,"4737":20,"4738":20,"4739":20,"4740":20,"4741":20,"4742":20,"4743":20,"4744":20,"4745":20,"4746":20,"4747":20,"4781":4,"4782":4,"4783":4,"4787":4,"4788":14,"4789":14,"4790":14,"4791":14,"4792":14,"4793":14,"4794":14,"4795":14,"4796":14,"4797":14,"4798":14,"4831":8,"4832":8,"4833":8,"4836":23,"4837":23,"4841":23,"4842":23,"4843":24,"4844":24,"4845":24,"4846":24,"4847":24,"4850":24,"4851":24,"4852":24,"4853":24,"4854":24,"4855":24,"4860":26,"4861":26,"4862":26,"4863":26,"4864":26,"4865":26,"4866":26,"4867":26,"4868":26,"4869":26,"4870":26,"4871":26,"4901":15,"4902":15,"4905":15,"4906":15,"4907":15,"4911":15,"4912":15,"4920":20,"4921":20,"4922":20,"4923":20,"4924":20,"4925":20,"4926":20,"4927":20,"4928":20,"4929":20,"4930":20,"4931":20,"4932":20,"4933":20,"4934":20,"4935":20,"4936":20,"4937":20,"4976":4,"4977":4,"4978":4,"4979":14,"4980":25,"4981":14,"4982":14,"4983":14,"4984":14,"4985":14,"4986":14,"4987":14,"4988":14,"5021":8,"5022":8,"5025":23,"5026":23,"5027":23,"5028":23,"5031":23,"5032":23,"5035":24,"5036":24,"5041":24,"5042":24,"5043":24,"5044":24,"5045":24,"5046":24,"5049":26,"5050":26,"5051":26,"5052":26,"5053":26,"5054":26,"5055":26,"5056":26,"5057":26,"5084":15,"5090":15,"5091":15,"5092":15,"5108":20,"5109":20,"5110":20,"5111":20,"5112":20,"5113":20,"5114":20,"5115":20,"5116":20,"5117":20,"5118":20,"5119":20,"5120":20,"5121":20,"5122":20,"5123":20,"5124":20,"5125":20,"5126":20,"5127":20,"5165":4,"5166":4,"5167":4,"5168":4,"5169":25,"5170":25,"5171":14,"5172":14,"5173":14,"5174":14,"5175":14,"5176":14,"5177":14,"5178":14,"5216":23,"5217":23,"5218":23,"5231":24,"5232":24,"5233":24,"5234":24,"5235":24,"5236":24,"5239":26,"5240":26,"5241":26,"5242":26,"5243":26,"5244":26,"5245":26,"5246":26,"5247":26,"5271":15,"5272":15,"5273":15,"5274":15,"5279":15,"5280":15,"5281":15,"5293":20,"5294":20,"5295":20,"5296":20,"5297":20,"5298":20,"5299":20,"5300":20,"5301":20,"5302":20,"5303":20,"5304":20,"5305":20,"5306":20,"5307":20,"5308":20,"5309":20,"5310":20,"5311":20,"5312":20,"5313":20,"5314":20,"5315":20,"5316":20,"5317":20,"5354":4,"5355":4,"5356":4,"5357":4,"5358":25,"5359":25,"5360":25,"5361":25,"5362":14,"5363":14,"5364":14,"5365":14,"5366":14,"5367":14,"5368":14,"5402":23,"5403":23,"5406":23,"5407":23,"5408":23,"5421":24,"5422":24,"5423":24,"5424":24,"5425":24,"5426":24,"5429":26,"5430":26,"5431":26,"5432":26,"5433":26,"5434":26,"5435":26,"5436":26,"5437":26,"5462":15,"5463":15,"5464":15,"5469":15,"5470":15,"5471":15,"5475":31,"5476":31,"5477":31,"5478":31,"5479":31,"5482":31,"5483":20,"5484":20,"5485":20,"5486":20,"5487":20,"5488":20,"5489":20,"5490":20,"5491":20,"5492":20,"5493":20,"5494":20,"5495":20,"5496":20,"5497":20,"5498":20,"5499":20,"5500":20,"5501":20,"5502":20,"5503":20,"5504":20,"5505":20,"5506":20,"5507":20,"5542":4,"5543":4,"5544":4,"5545":4,"5546":4,"5547":25,"5548":25,"5549":25,"5550":25,"5551":25,"5552":25,"5553":25,"5554":14,"5555":14,"5556":14,"5557":25,"5558":25,"5591":23,"5592":23,"5593":23,"5594":23,"5595":23,"5596":23,"5597":23,"5598":23,"5599":23,"5611":24,"5612":24,"5613":24,"5614":24,"5615":24,"5616":24,"5618":26,"5619":26,"5620":26,"5621":26,"5622":26,"5623":26,"5624":26,"5625":26,"5626":26,"5627":26,"5640":15,"5641":15,"5642":15,"5643":15,"5657":15,"5658":15,"5659":15,"5660":15,"5664":31,"5665":31,"5666":31,"5667":31,"5668":31,"5669":31,"5670":31,"5671":31,"5672":31,"5673":31,"5674":20,"5675":20,"5676":20,"5677":20,"5678":20,"5679":20,"5680":20,"5681":20,"5682":20,"5683":20,"5684":20,"5685":20,"5686":20,"5687":20,"5688":20,"5689":20,"5690":20,"5691":20,"5692":20,"5693":20,"5694":20,"5695":20,"5696":20,"5697":20,"5732":4,"5733":4,"5734":4,"5735":25,"5736":25,"5737":25,"5738":25,"5739":25,"5740":25,"5741":25,"5742":25,"5743":25,"5744":25,"5745":14,"5746":25,"5747":25,"5748":25,"5781":23,"5782":23,"5783":23,"5784":23,"5785":23,"5786":23,"5787":23,"5788":23,"5789":23,"5801":24,"5802":24,"5803":24,"5804":24,"5805":24,"5806":24,"5807":26,"5808":26,"5809":26,"5810":26,"5811":26,"5812":26,"5813":26,"5814":26,"5815":26,"5816":26,"5817":26,"5828":15,"5829":15,"5830":15,"5831":15,"5832":15,"5833":15,"5835":15,"5836":15,"5837":15,"5838":15,"5839":15,"5845":15,"5846":15,"5847":15,"5848":15,"5849":15,"5850":15,"5854":31,"5855":31,"5856":31,"5857":31,"5858":31,"5859":31,"5860":31,"5861":31,"5862":31,"5863":31,"5864":31,"5865":20,"5866":20,"5867":20,"5868":20,"5869":20,"5870":20,"5871":20,"5872":20,"5873":20,"5874":20,"5875":20,"5876":20,"5877":20,"5878":20,"5879":20,"5880":20,"5881":20,"5882":20,"5883":20,"5884":20,"5885":20,"5886":20,"5887":20,"5915":33,"5916":4,"5917":4,"5921":4,"5922":4,"5923":4,"5924":4,"5925":25,"5926":25,"5927":25,"5928":25,"5929":25,"5930":25,"5931":25,"5932":25,"5933":25,"5934":25,"5935":25,"5936":25,"5937":25,"5938":25,"5960":6,"5961":6,"5962":6,"5963":6,"5964":6,"5967":6,"5968":6,"5969":6,"5970":6,"5971":23,"5972":23,"5973":23,"5974":23,"5975":23,"5976":23,"5977":23,"5978":23,"5979":23,"5980":23,"5988":32,"5989":32,"5990":32,"5991":24,"5992":24,"5993":24,"5994":24,"5995":24,"5996":24,"5997":26,"5998":26,"5999":26,"6000":26,"6001":26,"6002":26,"6003":26,"6004":26,"6005":26,"6006":26,"6007":26,"6017":15,"6018":15,"6019":15,"6020":15,"6021":15,"6022":15,"6023":15,"6024":15,"6025":15,"6026":15,"6027":15,"6028":15,"6029":15,"6030":15,"6031":15,"6032":15,"6033":15,"6034":15,"6035":15,"6036":15,"6037":15,"6038":15,"6044":31,"6045":31,"6046":31,"6047":31,"6048":31,"6049":31,"6050":31,"6051":31,"6052":31,"6053":31,"6054":20,"6055":20,"6056":20,"6057":20,"6058":20,"6059":20,"6060":20,"6061":20,"6062":20,"6063":20,"6064":20,"6065":20,"6066":20,"6067":20,"6068":20,"6069":20,"6070":20,"6071":20,"6072":20,"6073":20,"6074":20,"6075":20,"6076":20,"6077":20,"6104":33,"6105":33,"6106":4,"6107":4,"6109":4,"6110":4,"6111":4,"6112":4,"6113":4,"6114":4,"6115":4,"6116":25,"6117":25,"6118":25,"6119":25,"6120":25,"6121":25,"6122":25,"6123":25,"6124":25,"6125":25,"6126":25,"6127":25,"6128":25,"6129":2,"6130":2,"6133":2,"6134":2,"6135":2,"6136":2,"6137":2,"6138":2,"6150":6,"6151":6,"6152":6,"6153":6,"6154":6,"6155":6,"6156":6,"6157":6,"6158":6,"6159":6,"6160":6,"6161":23,"6162":23,"6163":23,"6164":23,"6165":23,"6166":23,"6167":23,"6168":23,"6169":23,"6170":23,"6178":32,"6179":32,"6180":32,"6181":32,"6182":24,"6183":24,"6184":24,"6185":24,"6186":24,"6187":26,"6188":26,"6189":26,"6190":26,"6191":26,"6192":26,"6193":26,"6194":7,"6195":26,"6196":26,"6197":26,"6207":15,"6208":15,"6209":15,"6210":15,"6211":15,"6212":15,"6213":15,"6214":15,"6215":15,"6216":15,"6217":15,"6218":15,"6219":15,"6220":15,"6221":15,"6222":15,"6223":15,"6224":15,"6225":15,"6226":15,"6227":15,"6228":15,"6233":31,"6234":31,"6235":31,"6236":31,"6237":31,"6238":31,"6239":31,"6240":31,"6241":20,"6242":20,"6243":20,"6244":20,"6245":20,"6246":20,"6247":20,"6248":20,"6249":20,"6250":20,"6251":20,"6252":20,"6253":20,"6254":20,"6255":20,"6256":20,"6257":20,"6258":20,"6259":20,"6260":20,"6261":20,"6262":20,"6263":20,"6264":20,"6265":20,"6266":20,"6267":20,"6293":33,"6294":33,"6295":33,"6296":33,"6299":4,"6300":4,"6301":4,"6302":4,"6303":4,"6304":25,"6305":25,"6306":25,"6307":25,"6308":25,"6309":25,"6310":25,"6311":25,"6312":25,"6313":25,"6314":25,"6315":25,"6316":25,"6317":25,"6318":25,"6319":2,"6320":2,"6321":2,"6322":2,"6323":2,"6324":2,"6325":2,"6326":2,"6327":2,"6328":2,"6334":29,"6335":29,"6336":29,"6337":6,"6338":6,"6339":6,"6340":6,"6341":6,"6342":6,"6343":6,"6344":6,"6345":6,"6346":6,"6347":6,"6348":6,"6349":6,"6350":6,"6351":6,"6352":23,"6353":23,"6354":23,"6355":23,"6356":23,"6357":23,"6358":23,"6359":23,"6368":32,"6369":32,"6370":32,"6371":32,"6372":24,"6373":24,"6374":24,"6375":24,"6376":24,"6377":26,"6378":26,"6379":26,"6380":26,"6381":26,"6382":26,"6383":7,"6384":7,"6397":15,"6398":15,"6399":15,"6400":15,"6401":15,"6402":15,"6403":15,"6404":15,"6405":15,"6406":15,"6407":15,"6408":15,"6409":15,"6410":15,"6411":15,"6412":15,"6413":15,"6414":15,"6415":15,"6416":15,"6417":15,"6421":31,"6422":31,"6423":31,"6424":31,"6425":31,"6426":31,"6427":31,"6428":31,"6429":31,"6430":31,"6431":31,"6432":20,"6433":20,"6434":20,"6435":20,"6436":20,"6437":20,"6438":20,"6439":20,"6440":20,"6441":20,"6442":20,"6443":20,"6444":20,"6445":20,"6446":20,"6447":20,"6448":20,"6449":20,"6450":20,"6451":20,"6452":20,"6453":20,"6454":20,"6455":20,"6456":20,"6457":20,"6483":33,"6484":33,"6485":33,"6486":33,"6488":4,"6489":4,"6490":4,"6491":4,"6492":4,"6493":25,"6494":25,"6495":25,"6496":25,"6497":25,"6498":25,"6499":25,"6500":25,"6501":25,"6502":25,"6503":25,"6504":25,"6505":25,"6506":25,"6507":25,"6508":2,"6509":2,"6510":2,"6511":2,"6512":2,"6513":2,"6514":2,"6515":2,"6516":2,"6517":2,"6518":2,"6523":29,"6524":29,"6525":29,"6526":29,"6527":29,"6528":6,"6529":6,"6530":6,"6531":6,"6532":6,"6533":6,"6534":6,"6535":6,"6536":6,"6537":6,"6538":6,"6539":6,"6540":6,"6541":6,"6542":23,"6543":23,"6544":23,"6545":23,"6546":23,"6547":23,"6548":23,"6549":23,"6550":23,"6558":32,"6559":32,"6560":32,"6561":32,"6562":24,"6563":24,"6564":24,"6565":24,"6566":24,"6567":24,"6568":24,"6569":24,"6570":24,"6571":26,"6572":7,"6573":7,"6588":15,"6589":15,"6590":15,"6591":15,"6592":15,"6594":15,"6595":15,"6596":15,"6597":15,"6598":15,"6599":15,"6600":15,"6601":15,"6602":15,"6603":15,"6604":15,"6605":15,"6606":15,"6610":31,"6611":31,"6612":31,"6613":31,"6614":31,"6615":31,"6616":31,"6617":31,"6618":31,"6619":31,"6620":31,"6621":31,"6622":20,"6623":20,"6624":20,"6625":20,"6626":20,"6627":20,"6628":20,"6629":20,"6630":20,"6631":20,"6632":20,"6633":20,"6634":20,"6635":20,"6636":20,"6637":20,"6638":20,"6639":20,"6640":20,"6641":20,"6642":20,"6643":20,"6644":20,"6645":20,"6646":20,"6647":20,"6648":20,"6672":33,"6673":33,"6674":33,"6675":33,"6677":4,"6678":4,"6679":4,"6680":4,"6681":13,"6682":13,"6683":13,"6684":13,"6685":25,"6686":25,"6687":25,"6688":25,"6689":25,"6690":25,"6691":25,"6692":25,"6693":25,"6694":25,"6695":25,"6696":25,"6697":2,"6698":2,"6699":2,"6700":2,"6701":2,"6702":2,"6703":2,"6704":2,"6705":2,"6706":2,"6707":2,"6708":2,"6709":2,"6713":29,"6714":29,"6715":29,"6716":29,"6717":29,"6718":29,"6719":6,"6720":6,"6721":6,"6722":6,"6723":6,"6724":6,"6725":6,"6726":6,"6727":6,"6728":6,"6729":6,"6730":6,"6731":6,"6732":6,"6733":23,"6734":23,"6735":23,"6736":23,"6737":23,"6738":23,"6739":23,"6740":23,"6748":32,"6749":32,"6750":32,"6751":32,"6752":32,"6753":24,"6754":24,"6755":24,"6756":24,"6757":24,"6758":24,"6759":24,"6760":24,"6761":7,"6762":7,"6763":7,"6777":17,"6778":17,"6785":15,"6786":15,"6787":15,"6788":15,"6789":15,"6790":15,"6791":15,"6792":15,"6793":15,"6799":31,"6800":31,"6801":31,"6802":31,"6803":31,"6804":31,"6805":31,"6806":31,"6807":31,"6808":31,"6809":31,"6810":31,"6811":31,"6812":31,"6813":20,"6814":20,"6817":20,"6818":20,"6819":20,"6820":20,"6821":20,"6822":20,"6823":20,"6824":20,"6825":20,"6826":20,"6827":20,"6828":20,"6829":20,"6830":20,"6831":20,"6832":20,"6833":20,"6834":20,"6835":20,"6836":20,"6837":20,"6838":20,"6861":33,"6862":33,"6863":33,"6864":33,"6866":33,"6867":33,"6868":33,"6869":4,"6870":13,"6871":13,"6872":13,"6873":13,"6874":13,"6875":13,"6876":25,"6877":25,"6878":25,"6879":25,"6880":25,"6881":25,"6882":25,"6883":25,"6884":25,"6885":25,"6886":25,"6887":2,"6888":2,"6889":2,"6890":2,"6891":2,"6892":2,"6893":2,"6894":2,"6895":2,"6896":2,"6897":2,"6898":2,"6899":2,"6903":29,"6904":29,"6905":29,"6906":29,"6907":29,"6908":6,"6909":6,"6910":6,"6911":6,"6912":6,"6913":6,"6914":6,"6915":6,"6916":6,"6917":6,"6918":6,"6919":6,"6920":6,"6921":6,"6922":6,"6923":6,"6924":23,"6925":23,"6926":23,"6927":23,"6928":23,"6929":23,"6930":9,"6939":32,"6940":32,"6941":32,"6942":32,"6943":24,"6944":24,"6945":24,"6946":24,"6947":24,"6948":24,"6949":7,"6950":7,"6951":7,"6952":7,"6958":7,"6959":7,"6967":17,"6968":17,"6984":31,"6985":31,"6986":31,"6987":31,"6988":31,"6990":31,"6992":31,"6993":31,"6994":31,"6995":31,"6996":31,"6997":31,"6998":31,"6999":31,"7000":31,"7001":31,"7002":31,"7003":31,"7008":20,"7009":20,"7010":20,"7011":20,"7012":20,"7013":20,"7014":20,"7015":20,"7016":20,"7017":20,"7018":20,"7019":20,"7020":20,"7021":20,"7022":20,"7023":20,"7024":20,"7025":20,"7026":20,"7027":20,"7028":20,"7050":33,"7051":33,"7052":33,"7053":33,"7056":33,"7057":33,"7058":33,"7059":13,"7060":13,"7061":13,"7062":13,"7063":13,"7064":13,"7065":13,"7066":13,"7067":13,"7068":25,"7069":25,"7070":25,"7071":25,"7072":25,"7073":25,"7074":2,"7075":2,"7076":2,"7077":2,"7078":2,"7079":2,"7082":2,"7093":29,"7094":29,"7095":29,"7096":29,"7097":29,"7098":6,"7099":6,"7100":6,"7101":6,"7102":6,"7103":6,"7104":6,"7105":6,"7106":6,"7107":6,"7108":6,"7109":6,"7110":6,"7111":6,"7112":6,"7113":6,"7114":6,"7115":23,"7116":23,"7117":9,"7118":9,"7119":9,"7120":9,"7121":9,"7129":32,"7130":32,"7131":32,"7132":32,"7133":24,"7134":24,"7135":24,"7136":7,"7137":7,"7138":7,"7139":7,"7140":7,"7141":7,"7145":7,"7146":7,"7147":7,"7148":7,"7149":7,"7150":7,"7151":17,"7152":17,"7153":17,"7154":17,"7155":17,"7156":17,"7157":17,"7158":17,"7159":17,"7164":17,"7172":31,"7173":31,"7174":31,"7175":31,"7176":31,"7177":31,"7178":31,"7181":31,"7182":31,"7183":31,"7184":31,"7185":31,"7186":31,"7187":31,"7188":31,"7189":31,"7190":31,"7191":31,"7192":31,"7198":20,"7199":20,"7200":20,"7201":20,"7202":20,"7203":20,"7204":20,"7205":20,"7206":20,"7207":20,"7208":20,"7209":20,"7210":20,"7211":20,"7212":20,"7213":20,"7214":20,"7215":20,"7239":33,"7240":33,"7241":33,"7242":33,"7243":33,"7246":33,"7247":33,"7248":33,"7249":13,"7250":13,"7251":13,"7252":13,"7253":13,"7254":13,"7255":13,"7256":13,"7257":13,"7258":25,"7259":25,"7260":25,"7261":25,"7262":25,"7263":25,"7264":2,"7265":2,"7266":2,"7267":2,"7268":2,"7269":2,"7278":2,"7279":2,"7280":2,"7282":29,"7283":29,"7284":29,"7285":29,"7286":29,"7287":29,"7288":6,"7289":6,"7290":6,"7291":6,"7292":6,"7293":6,"7294":6,"7295":6,"7296":6,"7297":6,"7298":6,"7299":6,"7300":6,"7301":6,"7302":6,"7303":6,"7304":6,"7305":23,"7306":23,"7307":9,"7308":9,"7309":9,"7310":9,"7320":32,"7321":32,"7322":32,"7323":32,"7324":7,"7325":7,"7326":7,"7327":7,"7328":7,"7329":7,"7330":7,"7331":7,"7332":7,"7335":7,"7336":7,"7337":7,"7338":7,"7339":7,"7340":7,"7341":17,"7342":17,"7343":17,"7344":17,"7345":17,"7346":17,"7347":17,"7348":17,"7349":17,"7350":17,"7353":17,"7354":17,"7355":17,"7356":17,"7357":17,"7361":31,"7362":31,"7363":31,"7364":31,"7365":31,"7366":31,"7367":31,"7371":31,"7372":31,"7373":31,"7374":31,"7375":31,"7376":31,"7377":31,"7378":31,"7379":31,"7380":31,"7381":31,"7382":31,"7385":20,"7386":20,"7387":20,"7388":20,"7389":20,"7390":20,"7391":20,"7392":20,"7393":20,"7394":20,"7395":20,"7396":20,"7397":20,"7398":20,"7399":20,"7400":20,"7401":20,"7428":33,"7429":33,"7430":33,"7431":33,"7435":33,"7436":33,"7437":33,"7438":33,"7439":33,"7440":13,"7441":13,"7442":13,"7443":13,"7444":13,"7445":13,"7446":13,"7447":13,"7448":13,"7449":13,"7450":13,"7451":13,"7452":25,"7454":2,"7455":2,"7456":2,"7457":2,"7458":2,"7468":2,"7469":2,"7470":2,"7473":29,"7474":29,"7475":29,"7476":29,"7477":29,"7478":6,"7479":6,"7480":6,"7481":6,"7482":6,"7483":6,"7484":6,"7485":6,"7486":6,"7487":6,"7488":6,"7489":6,"7490":6,"7491":6,"7492":6,"7493":6,"7494":6,"7495":23,"7496":23,"7497":9,"7498":9,"7499":9,"7500":9,"7501":9,"7510":32,"7511":32,"7512":32,"7513":7,"7514":7,"7515":7,"7516":7,"7517":7,"7518":7,"7519":7,"7520":7,"7521":7,"7522":7,"7523":7,"7524":7,"7525":7,"7526":7,"7527":7,"7528":7,"7529":7,"7530":17,"7531":17,"7532":17,"7533":17,"7534":17,"7535":17,"7536":17,"7537":17,"7538":17,"7539":17,"7540":17,"7542":17,"7543":17,"7544":17,"7545":17,"7546":17,"7547":17,"7548":17,"7549":17,"7551":31,"7552":31,"7553":31,"7554":31,"7555":31,"7556":31,"7557":31,"7558":31,"7559":31,"7560":31,"7561":31,"7562":31,"7563":31,"7564":31,"7565":31,"7566":31,"7567":31,"7568":31,"7569":31,"7570":31,"7571":31,"7573":20,"7574":20,"7575":20,"7576":20,"7577":20,"7578":20,"7579":20,"7580":20,"7581":20,"7582":20,"7583":20,"7584":20,"7585":20,"7586":20,"7587":20,"7588":20,"7589":20,"7617":33,"7618":33,"7619":33,"7620":33,"7621":33,"7624":33,"7625":33,"7626":33,"7627":33,"7628":33,"7629":33,"7630":33,"7631":13,"7632":13,"7633":13,"7634":13,"7635":13,"7636":13,"7637":13,"7638":13,"7639":13,"7640":13,"7641":13,"7642":13,"7645":2,"7646":2,"7647":2,"7648":2,"7658":2,"7659":2,"7660":2,"7661":29,"7662":29,"7663":29,"7664":29,"7665":29,"7666":29,"7667":29,"7668":29,"7669":6,"7670":6,"7671":6,"7672":6,"7673":6,"7674":6,"7675":6,"7676":6,"7677":6,"7678":6,"7679":6,"7680":6,"7681":6,"7682":6,"7683":6,"7684":6,"7685":6,"7686":6,"7687":9,"7688":9,"7689":9,"7690":9,"7691":9,"7692":9,"7700":32,"7701":32,"7702":7,"7703":7,"7704":7,"7705":7,"7706":7,"7707":7,"7708":7,"7709":7,"7710":7,"7711":7,"7712":7,"7713":7,"7714":7,"7715":7,"7716":7,"7717":7,"7718":7,"7721":17,"7722":17,"7732":17,"7733":17,"7734":17,"7735":17,"7736":17,"7737":17,"7738":17,"7739":17,"7740":17,"7741":31,"7742":31,"7743":31,"7744":31,"7745":31,"7746":31,"7747":31,"7748":31,"7749":31,"7750":31,"7751":31,"7752":31,"7753":31,"7754":31,"7755":31,"7756":31,"7757":31,"7758":31,"7759":31,"7760":31,"7761":31,"7763":20,"7764":20,"7765":20,"7766":20,"7767":20,"7768":20,"7769":20,"7770":20,"7771":20,"7773":20,"7774":20,"7775":20,"7776":20,"7778":20,"7779":20,"7807":33,"7808":33,"7809":33,"7810":33,"7814":33,"7815":33,"7816":33,"7817":33,"7818":33,"7819":33,"7820":33,"7821":13,"7822":13,"7823":13,"7824":13,"7825":13,"7826":13,"7827":13,"7828":13,"7829":13,"7830":13,"7831":13,"7832":13,"7850":2,"7851":29,"7852":29,"7853":29,"7854":29,"7855":29,"7856":29,"7857":29,"7858":29,"7859":29,"7860":6,"7861":6,"7862":6,"7863":6,"7864":6,"7865":6,"7866":6,"7867":6,"7868":6,"7869":6,"7870":6,"7871":6,"7872":6,"7873":6,"7874":6,"7875":6,"7876":6,"7877":9,"7878":9,"7879":9,"7880":9,"7881":9,"7882":9,"7883":9,"7884":9,"7890":32,"7891":32,"7892":7,"7893":7,"7894":7,"7895":7,"7896":7,"7897":7,"7898":7,"7899":7,"7900":7,"7901":7,"7902":7,"7903":7,"7904":7,"7905":7,"7906":7,"7907":7,"7908":7,"7922":17,"7923":17,"7924":17,"7925":17,"7926":17,"7927":17,"7928":17,"7929":17,"7930":31,"7931":31,"7932":31,"7933":31,"7934":31,"7935":31,"7936":31,"7937":31,"7938":31,"7939":31,"7940":31,"7941":31,"7942":31,"7943":31,"7944":31,"7945":31,"7946":31,"7947":31,"7948":31,"7949":31,"7950":31,"7951":31,"7952":20,"7953":20,"7954":20,"7955":20,"7956":20,"7957":20,"7958":20,"7959":20,"7960":20,"7961":20,"7997":19,"7998":33,"7999":33,"8000":33,"8004":33,"8005":33,"8006":33,"8007":33,"8008":33,"8009":33,"8010":21,"8011":21,"8012":21,"8013":21,"8014":21,"8015":21,"8016":13,"8017":13,"8018":13,"8019":13,"8020":13,"8021":13,"8022":22,"8040":29,"8041":29,"8042":29,"8043":29,"8044":29,"8045":29,"8046":29,"8047":29,"8048":29,"8049":29,"8050":29,"8051":29,"8052":6,"8053":6,"8054":6,"8055":6,"8056":6,"8057":6,"8058":6,"8059":6,"8060":6,"8061":6,"8062":6,"8063":6,"8064":6,"8065":6,"8066":9,"8067":9,"8068":9,"8069":9,"8070":9,"8071":9,"8072":9,"8073":9,"8074":9,"8075":9,"8081":7,"8082":7,"8083":7,"8084":7,"8085":7,"8086":7,"8088":7,"8089":7,"8090":7,"8091":7,"8092":7,"8093":7,"8094":7,"8095":7,"8096":7,"8097":7,"8098":7,"8112":17,"8113":17,"8114":17,"8115":17,"8116":17,"8117":17,"8118":17,"8121":31,"8122":31,"8123":31,"8124":31,"8125":31,"8126":31,"8127":31,"8128":31,"8129":31,"8130":31,"8131":31,"8132":31,"8133":31,"8134":31,"8135":31,"8136":31,"8137":31,"8138":31,"8139":31,"8140":31,"8141":31,"8142":20,"8143":20,"8144":20,"8145":20,"8146":20,"8147":20,"8148":20,"8149":20,"8150":20,"8186":19,"8187":19,"8188":19,"8189":33,"8193":33,"8194":33,"8195":33,"8196":33,"8197":33,"8198":33,"8199":21,"8200":21,"8201":21,"8202":21,"8203":21,"8204":21,"8205":21,"8206":21,"8207":21,"8208":21,"8209":21,"8210":22,"8211":22,"8212":22,"8213":22,"8229":29,"8230":29,"8231":29,"8232":29,"8233":29,"8234":29,"8235":29,"8236":29,"8237":29,"8238":29,"8239":29,"8240":29,"8241":29,"8242":29,"8243":29,"8244":29,"8245":6,"8246":6,"8247":6,"8248":6,"8249":6,"8250":6,"8251":6,"8252":6,"8253":6,"8254":6,"8255":9,"8256":9,"8257":9,"8258":9,"8259":9,"8260":9,"8261":9,"8262":9,"8263":9,"8264":9,"8265":9,"8271":7,"8272":7,"8273":7,"8274":7,"8275":7,"8279":7,"8280":7,"8281":7,"8282":7,"8283":7,"8284":7,"8285":7,"8286":7,"8287":7,"8288":7,"8301":17,"8302":17,"8303":17,"8304":17,"8305":17,"8306":17,"8307":17,"8308":17,"8309":17,"8312":31,"8313":31,"8314":31,"8315":31,"8316":31,"8317":31,"8318":31,"8319":31,"8320":31,"8321":31,"8322":31,"8323":31,"8324":31,"8325":31,"8326":31,"8327":31,"8328":31,"8329":31,"8330":31,"8334":20,"8335":20,"8336":20,"8337":20,"8338":20,"8375":19,"8376":19,"8377":19,"8378":19,"8382":33,"8383":33,"8384":33,"8385":33,"8386":33,"8387":33,"8388":21,"8389":21,"8390":21,"8391":21,"8392":21,"8393":21,"8394":21,"8395":21,"8396":21,"8397":21,"8398":21,"8399":21,"8400":22,"8401":22,"8402":22,"8403":22,"8404":22,"8419":29,"8420":29,"8421":29,"8422":29,"8423":29,"8424":29,"8425":29,"8426":29,"8427":29,"8428":29,"8429":29,"8430":29,"8431":29,"8432":29,"8433":29,"8434":29,"8435":29,"8436":29,"8437":6,"8438":6,"8439":6,"8440":6,"8441":6,"8442":6,"8443":6,"8444":6,"8445":9,"8446":9,"8447":9,"8448":9,"8449":9,"8450":9,"8451":9,"8452":9,"8453":9,"8454":9,"8455":9,"8462":7,"8463":7,"8464":7,"8469":7,"8470":7,"8471":7,"8472":7,"8473":7,"8474":7,"8475":7,"8476":7,"8477":7,"8491":17,"8492":17,"8493":17,"8494":17,"8495":17,"8496":17,"8497":17,"8498":17,"8501":17,"8502":17,"8503":31,"8504":31,"8505":31,"8506":31,"8507":31,"8508":31,"8509":31,"8510":31,"8511":31,"8512":31,"8513":31,"8514":31,"8515":31,"8516":31,"8517":31,"8524":20,"8525":20,"8526":20,"8564":19,"8565":19,"8566":19,"8567":19,"8568":19,"8569":19,"8571":33,"8572":33,"8573":33,"8574":33,"8575":33,"8576":33,"8577":21,"8578":21,"8579":21,"8580":21,"8581":21,"8582":21,"8583":21,"8584":21,"8585":21,"8586":21,"8587":21,"8588":21,"8589":21,"8590":22,"8591":22,"8592":22,"8593":22,"8594":22,"8602":22,"8603":22,"8608":29,"8609":29,"8610":29,"8611":29,"8612":29,"8613":29,"8614":29,"8615":29,"8616":29,"8617":29,"8618":29,"8619":29,"8620":29,"8621":29,"8622":29,"8623":29,"8624":29,"8625":29,"8626":29,"8627":29,"8628":6,"8629":6,"8630":6,"8631":6,"8632":6,"8633":6,"8634":6,"8635":9,"8636":9,"8637":9,"8638":9,"8639":9,"8640":9,"8641":9,"8642":9,"8643":9,"8644":9,"8651":7,"8652":7,"8653":7,"8654":7,"8655":7,"8659":7,"8660":7,"8661":7,"8662":7,"8663":7,"8664":7,"8681":17,"8682":17,"8683":17,"8684":17,"8685":17,"8686":17,"8690":17,"8691":17,"8692":17,"8693":31,"8694":31,"8695":31,"8696":31,"8697":31,"8698":31,"8699":31,"8702":31,"8703":31,"8704":31,"8705":31,"8715":20,"8754":19,"8755":19,"8756":19,"8757":19,"8758":19,"8759":19,"8760":33,"8761":33,"8762":33,"8763":33,"8764":33,"8765":33,"8766":33,"8767":21,"8768":21,"8769":21,"8770":21,"8771":21,"8772":21,"8773":21,"8774":21,"8775":21,"8776":21,"8777":21,"8778":21,"8779":21,"8780":22,"8781":22,"8782":22,"8783":22,"8792":22,"8793":22,"8797":29,"8798":29,"8799":29,"8800":29,"8801":29,"8802":29,"8803":29,"8804":29,"8805":29,"8806":29,"8807":29,"8808":29,"8809":29,"8810":29,"8811":29,"8812":29,"8813":29,"8814":29,"8815":29,"8816":29,"8817":29,"8818":6,"8819":6,"8820":6,"8821":6,"8822":6,"8823":9,"8824":9,"8825":9,"8826":9,"8827":9,"8828":9,"8829":9,"8830":9,"8831":9,"8832":9,"8833":9,"8834":9,"8841":7,"8842":7,"8843":7,"8844":7,"8845":7,"8847":7,"8848":7,"8850":7,"8851":11,"8856":11,"8857":11,"8858":18,"8859":18,"8860":18,"8861":18,"8871":17,"8872":17,"8873":17,"8874":17,"8875":17,"8876":17,"8877":17,"8878":17,"8879":17,"8880":17,"8881":17,"8882":17,"8883":31,"8884":31,"8885":31,"8886":31,"8887":31,"8888":31,"8889":31,"8944":19,"8945":19,"8947":19,"8948":19,"8949":19,"8950":19,"8951":33,"8952":33,"8953":33,"8954":33,"8955":33,"8956":21,"8957":21,"8958":21,"8959":21,"8960":21,"8961":21,"8962":21,"8963":21,"8964":21,"8965":21,"8966":21,"8967":21,"8968":21,"8969":21,"8970":22,"8971":22,"8972":22,"8982":22,"8983":22,"8988":29,"8989":29,"8990":29,"8991":29,"8992":29,"8993":29,"8994":29,"8995":29,"8996":29,"8997":29,"8998":29,"8999":29,"9000":29,"9001":29,"9002":29,"9003":29,"9004":29,"9005":29,"9006":29,"9007":29,"9008":6,"9009":6,"9010":6,"9011":6,"9012":6,"9013":6,"9014":9,"9015":9,"9016":9,"9017":9,"9018":9,"9019":9,"9020":9,"9021":9,"9022":9,"9023":9,"9024":9,"9025":9,"9032":7,"9033":7,"9034":7,"9035":7,"9036":7,"9037":7,"9038":7,"9039":11,"9040":11,"9041":11,"9042":11,"9043":11,"9044":11,"9045":11,"9046":11,"9047":11,"9048":18,"9049":18,"9050":18,"9051":18,"9052":18,"9062":17,"9063":17,"9064":17,"9065":17,"9066":17,"9067":17,"9068":17,"9069":17,"9070":17,"9072":31,"9073":31,"9074":31,"9075":31,"9077":31,"9078":31,"9079":31,"9132":19,"9133":19,"9134":19,"9135":19,"9136":19,"9139":19,"9140":19,"9141":19,"9142":19,"9143":19,"9144":33,"9145":21,"9146":21,"9147":21,"9148":21,"9149":21,"9150":21,"9151":21,"9152":21,"9153":21,"9154":21,"9155":21,"9156":21,"9157":22,"9158":22,"9159":22,"9160":22,"9161":22,"9162":22,"9163":22,"9174":22,"9178":29,"9179":29,"9180":29,"9181":29,"9182":29,"9183":29,"9184":29,"9185":29,"9186":29,"9187":29,"9188":29,"9189":29,"9190":29,"9191":29,"9192":29,"9193":29,"9194":29,"9195":29,"9196":29,"9197":29,"9198":29,"9199":6,"9200":6,"9201":9,"9202":6,"9203":6,"9204":9,"9205":9,"9206":9,"9207":9,"9208":9,"9209":9,"9210":9,"9211":9,"9212":9,"9213":9,"9214":9,"9215":9,"9216":9,"9217":9,"9218":9,"9219":9,"9223":7,"9224":7,"9225":7,"9226":7,"9227":7,"9228":7,"9229":11,"9230":11,"9231":11,"9232":11,"9233":11,"9234":11,"9235":11,"9236":11,"9237":11,"9238":18,"9239":18,"9240":18,"9241":18,"9242":18,"9243":18,"9252":17,"9253":17,"9254":17,"9255":17,"9256":17,"9257":17,"9258":17,"9259":17,"9260":17,"9263":31,"9267":31,"9268":31,"9269":31,"9280":20,"9281":20,"9322":19,"9323":19,"9324":19,"9325":19,"9326":19,"9329":19,"9330":19,"9331":19,"9332":19,"9333":19,"9334":19,"9335":21,"9336":21,"9337":21,"9338":21,"9339":21,"9340":21,"9341":21,"9342":21,"9343":21,"9344":21,"9345":21,"9346":21,"9347":22,"9348":22,"9349":22,"9350":22,"9351":22,"9352":22,"9353":22,"9362":22,"9363":22,"9364":22,"9365":22,"9368":29,"9369":29,"9370":29,"9371":29,"9372":29,"9373":29,"9374":29,"9375":29,"9376":29,"9377":29,"9378":29,"9379":29,"9380":29,"9381":29,"9382":29,"9383":29,"9384":29,"9385":29,"9386":29,"9387":29,"9388":29,"9389":29,"9390":9,"9391":9,"9392":9,"9393":9,"9394":9,"9395":9,"9396":9,"9397":9,"9398":9,"9399":9,"9400":9,"9401":9,"9402":9,"9403":9,"9404":9,"9405":9,"9406":9,"9407":9,"9408":9,"9409":9,"9410":9,"9413":7,"9414":7,"9415":7,"9416":7,"9417":7,"9418":7,"9419":7,"9420":11,"9421":11,"9422":11,"9423":11,"9424":11,"9425":11,"9426":11,"9427":18,"9428":18,"9429":18,"9430":18,"9431":18,"9432":18,"9433":18,"9434":18,"9437":17,"9438":17,"9439":17,"9442":17,"9443":17,"9444":17,"9445":17,"9446":17,"9447":17,"9448":17,"9449":17,"9458":31,"9459":31,"9470":20,"9471":20,"9511":19,"9512":19,"9513":19,"9514":19,"9515":19,"9516":19,"9518":19,"9519":19,"9520":19,"9521":19,"9522":19,"9523":19,"9524":19,"9525":21,"9526":21,"9527":21,"9528":21,"9529":21,"9530":21,"9531":21,"9532":21,"9533":21,"9534":21,"9535":21,"9536":21,"9537":22,"9538":22,"9539":22,"9540":22,"9541":22,"9542":22,"9543":22,"9552":22,"9553":22,"9554":22,"9558":29,"9559":29,"9560":29,"9561":29,"9562":29,"9563":29,"9564":29,"9565":29,"9566":29,"9567":29,"9568":29,"9569":29,"9570":29,"9571":29,"9572":29,"9573":29,"9574":29,"9575":29,"9576":29,"9577":29,"9578":29,"9579":29,"9580":9,"9581":9,"9582":9,"9583":9,"9584":16,"9585":9,"9586":9,"9587":9,"9588":9,"9589":9,"9590":9,"9591":9,"9592":9,"9593":9,"9594":9,"9595":9,"9596":9,"9597":9,"9598":9,"9599":9,"9600":9,"9604":7,"9605":7,"9606":7,"9607":7,"9608":7,"9609":7,"9610":7,"9611":7,"9612":7,"9613":11,"9614":11,"9615":11,"9616":18,"9617":18,"9618":18,"9619":18,"9620":18,"9621":18,"9622":18,"9623":18,"9624":18,"9627":17,"9628":17,"9629":17,"9630":17,"9632":17,"9633":17,"9634":17,"9635":17,"9636":17,"9637":17,"9638":17,"9639":17,"9648":31,"9649":31,"9661":20,"9701":19,"9702":19,"9703":19,"9704":19,"9705":19,"9707":19,"9708":19,"9709":19,"9710":19,"9711":19,"9712":19,"9713":19,"9714":19,"9715":21,"9716":21,"9717":21,"9718":21,"9719":21,"9720":21,"9721":21,"9722":21,"9723":21,"9724":21,"9725":21,"9726":22,"9727":22,"9729":22,"9730":22,"9731":22,"9732":22,"9743":22,"9749":29,"9750":29,"9751":29,"9752":29,"9753":29,"9754":29,"9755":29,"9758":29,"9761":29,"9762":29,"9763":29,"9764":29,"9765":29,"9766":29,"9767":29,"9768":29,"9769":29,"9770":29,"9771":9,"9772":9,"9773":16,"9774":16,"9775":16,"9776":16,"9777":9,"9778":9,"9779":9,"9780":9,"9781":9,"9782":9,"9783":9,"9784":9,"9785":9,"9786":9,"9787":9,"9788":9,"9789":9,"9795":7,"9796":7,"9797":7,"9798":7,"9799":7,"9800":7,"9801":7,"9811":18,"9812":18,"9813":18,"9814":18,"9815":18,"9818":17,"9819":17,"9820":17,"9823":17,"9824":17,"9825":17,"9826":17,"9827":17,"9828":17,"9829":17,"9891":19,"9892":19,"9893":19,"9894":19,"9897":19,"9898":19,"9899":19,"9900":19,"9901":19,"9902":19,"9903":19,"9904":19,"9905":19,"9906":21,"9907":21,"9908":21,"9909":21,"9910":21,"9911":21,"9912":21,"9913":21,"9914":21,"9939":29,"9940":29,"9941":29,"9942":29,"9943":29,"9944":29,"9952":29,"9953":29,"9954":29,"9955":29,"9956":29,"9957":29,"9959":29,"9960":29,"9961":9,"9962":16,"9963":16,"9964":16,"9965":16,"9966":16,"9967":16,"9968":9,"9969":9,"9970":9,"9971":9,"9972":9,"9973":9,"9974":9,"9975":9,"9976":9,"9977":9,"9978":9,"9987":7,"10002":18,"10003":18,"10004":18,"10005":18,"10009":17,"10013":17,"10014":17,"10015":17,"10016":17,"10017":17,"10018":17,"10019":17,"10080":0,"10081":0,"10082":0,"10083":0,"10086":19,"10087":19,"10088":19,"10089":19,"10090":19,"10091":19,"10092":19,"10093":19,"10094":19,"10095":21,"10096":21,"10097":21,"10098":21,"10099":21,"10100":21,"10101":21,"10102":21,"10130":29,"10131":29,"10132":29,"10133":29,"10145":29,"10152":16,"10153":16,"10154":16,"10155":16,"10156":16,"10157":16,"10158":16,"10159":9,"10160":9,"10161":9,"10162":9,"10163":9,"10164":9,"10165":9,"10166":9,"10167":9,"10168":9,"10192":18,"10193":18,"10194":18,"10195":18,"10203":17,"10204":17,"10205":17,"10206":17,"10207":17,"10208":17,"10268":0,"10269":0,"10270":0,"10271":0,"10272":0,"10273":0,"10274":0,"10275":0,"10276":19,"10277":19,"10278":19,"10279":19,"10280":19,"10281":19,"10282":19,"10283":19,"10284":19,"10285":21,"10286":21,"10287":21,"10288":21,"10289":21,"10290":21,"10291":21,"10320":29,"10321":29,"10322":29,"10323":29,"10342":16,"10343":16,"10344":16,"10345":16,"10346":16,"10347":16,"10348":16,"10349":9,"10350":9,"10351":9,"10352":9,"10353":9,"10354":9,"10355":9,"10356":9,"10357":9,"10358":9,"10359":9,"10384":18,"10385":18,"10386":18,"10394":17,"10395":17,"10396":17,"10397":17,"10398":17,"10399":17,"10457":0,"10458":0,"10459":0,"10460":0,"10461":0,"10462":0,"10463":0,"10464":0,"10465":0,"10466":0,"10467":19,"10468":19,"10469":19,"10470":19,"10471":19,"10472":19,"10473":19,"10474":19,"10475":21,"10476":21,"10477":21,"10478":21,"10480":21,"10509":22,"10510":22,"10511":22,"10512":29,"10532":16,"10533":16,"10534":16,"10535":16,"10536":16,"10537":16,"10538":16,"10539":16,"10540":9,"10541":9,"10542":9,"10543":9,"10544":9,"10545":9,"10546":9,"10547":9,"10548":9,"10549":9,"10574":18,"10575":18,"10576":18,"10584":17,"10585":17,"10586":17,"10587":17,"10588":17,"10589":17,"10646":0,"10647":0,"10648":0,"10649":0,"10650":0,"10653":0,"10654":0,"10655":0,"10656":19,"10657":19,"10658":19,"10659":19,"10660":19,"10661":19,"10662":19,"10663":19,"10664":19,"10665":19,"10686":22,"10687":22,"10688":22,"10689":22,"10698":22,"10699":22,"10700":22,"10701":22,"10723":16,"10724":16,"10725":16,"10726":16,"10727":16,"10728":16,"10729":16,"10730":16,"10731":16,"10732":16,"10733":16,"10734":16,"10735":16,"10736":9,"10737":9,"10764":18,"10765":18,"10766":18,"10776":17,"10777":17,"10778":17,"10779":17,"10834":0,"10835":0,"10836":0,"10837":0,"10838":0,"10839":0,"10842":0,"10843":0,"10844":0,"10845":0,"10846":19,"10847":19,"10848":19,"10849":19,"10850":19,"10851":19,"10852":19,"10853":19,"10854":19,"10855":19,"10875":22,"10876":22,"10877":22,"10878":22,"10879":22,"10880":22,"10884":22,"10885":22,"10888":22,"10889":22,"10890":22,"10913":16,"10914":16,"10915":16,"10916":16,"10917":16,"10918":16,"10919":16,"10920":16,"10921":16,"10922":16,"10923":16,"10924":16,"10925":16,"10954":18,"10955":18,"10956":18,"10967":17,"10968":17,"11024":0,"11025":0,"11026":0,"11027":0,"11032":0,"11033":0,"11034":0,"11035":0,"11036":19,"11037":19,"11038":19,"11039":19,"11040":19,"11041":19,"11042":19,"11043":19,"11044":19,"11065":22,"11066":22,"11067":22,"11068":22,"11069":22,"11070":22,"11074":22,"11075":22,"11078":22,"11079":22,"11080":22,"11104":16,"11105":16,"11106":16,"11107":16,"11108":16,"11109":16,"11110":16,"11111":16,"11112":16,"11113":16,"11114":16,"11115":16,"11145":18,"11146":18,"11221":0,"11222":0,"11223":0,"11224":0,"11225":0,"11226":19,"11227":19,"11228":19,"11229":19,"11230":19,"11231":19,"11232":19,"11233":19,"11256":22,"11257":22,"11258":22,"11259":22,"11260":22,"11265":22,"11266":22,"11267":22,"11295":16,"11296":16,"11297":16,"11298":16,"11299":16,"11300":16,"11301":16,"11302":16,"11303":16,"11304":16,"11305":16,"11306":16,"11335":18,"11336":18,"11337":18,"11409":0,"11410":0,"11411":0,"11412":0,"11413":0,"11414":0,"11415":19,"11416":19,"11417":19,"11418":19,"11419":19,"11420":19,"11421":19,"11448":22,"11449":22,"11450":22,"11455":22,"11456":22,"11457":22,"11458":22,"11485":16,"11486":16,"11487":16,"11488":16,"11489":16,"11490":16,"11491":16,"11492":16,"11493":16,"11494":16,"11495":16,"11496":16,"11524":18,"11525":18,"11526":18,"11527":18,"11530":18,"11531":18,"11598":0,"11599":0,"11600":0,"11601":0,"11602":0,"11603":0,"11604":0,"11605":0,"11606":19,"11607":19,"11608":19,"11609":19,"11610":19,"11644":22,"11645":22,"11646":22,"11647":22,"11648":22,"11675":16,"11676":16,"11677":16,"11678":16,"11679":16,"11680":16,"11681":16,"11682":16,"11683":16,"11684":16,"11685":16,"11686":16,"11715":18,"11716":18,"11717":18,"11719":18,"11720":18,"11721":18,"11722":18,"11787":0,"11788":0,"11789":0,"11790":0,"11791":0,"11792":0,"11793":0,"11794":0,"11795":0,"11796":19,"11797":19,"11798":19,"11799":19,"11834":22,"11835":22,"11836":22,"11837":22,"11838":22,"11865":16,"11866":16,"11867":16,"11868":16,"11869":16,"11870":16,"11871":16,"11872":16,"11873":16,"11874":16,"11875":16,"11876":16,"11904":18,"11905":18,"11909":18,"11910":18,"11911":18,"11912":18,"11976":0,"11977":0,"11978":0,"11979":0,"11980":0,"11981":0,"11982":0,"11983":0,"11984":0,"11985":0,"11986":0,"11987":0,"12025":22,"12026":22,"12027":22,"12056":16,"12057":16,"12058":16,"12059":16,"12060":16,"12061":16,"12062":16,"12063":16,"12064":16,"12065":16,"12094":18,"12095":18,"12100":18,"12101":18,"12102":18,"12165":0,"12166":0,"12167":0,"12168":0,"12169":0,"12170":0,"12171":0,"12172":0,"12173":0,"12174":0,"12175":0,"12176":0,"12177":0,"12215":22,"12216":22,"12247":16,"12248":16,"12249":16,"12250":16,"12251":16,"12252":16,"12253":16,"12285":18,"12286":18,"12290":18,"12291":18,"12292":18,"12293":18,"12354":0,"12355":0,"12356":0,"12357":0,"12358":0,"12359":0,"12360":0,"12361":0,"12362":0,"12363":0,"12364":0,"12365":0,"12366":0,"12405":22,"12406":22,"12475":18,"12476":18,"12481":18,"12482":18,"12483":18,"12544":0,"12545":0,"12546":0,"12547":0,"12548":0,"12549":0,"12550":0,"12551":0,"12552":0,"12553":0,"12554":0,"12555":0,"12734":0,"12735":0,"12736":0,"12737":0,"12738":0,"12739":0,"12740":0,"12741":0,"12742":0,"12743":0,"12744":0,"12859":18,"12860":18,"12923":0,"12924":0,"12925":0,"12926":0,"12927":0,"12928":0,"13049":18,"13050":18,"13051":18,"13113":0,"13114":0,"13115":0,"13116":0,"13304":0,"13305":0},"sumber":"geoBoundaries IDN ADM1 (CC BY 4.0), www.geoboundaries.org"}

## 3. Uji koneksi (satu file)

In [ ]:
import json, importlib, harmonisasi as h
importlib.reload(h)
cells = json.load(open("cells.json"))
dl = h.Pengunduh(MAP_KEY, cache="cache")
uji = dl.tahunan("MODIS", 2019)
print("Arsip tahunan MODIS 2019:", "OK, %d baris" % uji.count("\n") if uji else "GAGAL (akan memakai API)")
uji2 = dl.api("VIIRS_NOAA20_NRT", __import__("datetime").date.today() - __import__("datetime").timedelta(days=3), 1)
print("FIRMS API (NOAA-20 NRT):", "OK, %d baris" % uji2.count("\n") if uji2 else "GAGAL — cek MAP_KEY")

## 4. Jalankan semuanya
Unduhan pertama ±10–30 menit. Hasil per tahun disimpan di folder `cache/`, jadi kalau terputus cukup jalankan ulang sel ini.

In [ ]:
lebar, ktab, info, val, pred = h.jalankan(MAP_KEY, cells, keluar="web/data", cache="cache", overlap=(2012, 2021))

## 5. Lihat hasil

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
nas = lebar.groupby("m")[["M", "V", "H"]].sum()
nas.index = pd.to_datetime([h.label_bulan(i) for i in nas.index])
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(nas.index, nas["V"], color="#bbbbbb", lw=1, label="VIIRS mentah")
ax.plot(nas.index, nas["M"], color="#1f77b4", lw=1.2, label="MODIS mentah (Terra+Aqua)")
ax.plot(nas.index, nas["H"], color="#d62728", lw=1.6, label="Harmonis (setara MODIS)")
ax.set_ylabel("titik api / bulan"); ax.set_title("Indonesia — titik api bulanan"); ax.legend(); plt.show()

In [ ]:
print("Faktor k per pulau × bulan (ΣMODIS/ΣVIIRS, tumpang tindih 2012–2021):")
display(ktab.pivot(index="pulau", columns="bulan", values="k").round(3))
print("\nValidasi leave-one-year-out:")
display(pd.DataFrame({k: {**{f"harmonis_{a}": b for a, b in v["harmonis"].items()},
                          **{f"mentah_{a}": b for a, b in v["viirs_mentah"].items()}, "n_bulan": v["n_bulan"]}
                      for k, v in val.items() if v}).T)

## 6. Unduh data untuk web app
Ekstrak `data_web.zip` ke folder `docs/data/` di repo GitHub.

In [ ]:
import shutil
shutil.make_archive("data_web", "zip", "web/data")
try:
    from google.colab import files
    files.download("data_web.zip")
except Exception:
    print("Unduh manual: panel Files (kiri) → data_web.zip")